[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/05_collector/05_proyecto_collector.ipynb)

# Proyecto 05 · Configura un Collector que reciba OTLP y exporte trazas, métricas y logs a los tres backends

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Opcional `docker compose --profile core --profile metrics up -d` para el último control (R11). **Sin el stack** funcionan R1–R10 (con Docker) y R1–R8 sin Docker (solo linter y análisis del YAML) |
| **RAM aproximada** | ≈ 0,1 GB por Collector de laboratorio; con el stack ≈ 0,5 GB |
| **Prerrequisitos** | Lección `05_collector` (componentes, orden, `validate`, linter, métricas internas) y módulos 03-04 |
| **Conexión** | Sustituyes el Collector del stack por **el tuyo**: aplica las políticas que se diseñaron en 03 (SDK), 04 (spec GenAI y PII) y el siguiente paso natural para los módulos 18 (muestreo) y 20 (privacidad) |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. Seguridad y Plataforma exigen un **gateway** único de telemetría con estas políticas, aplicadas **sin tocar las aplicaciones**:

| # | Requisito | Detalle |
|---|---|---|
| A | Recibir OTLP | gRPC `0.0.0.0:4317` y HTTP `0.0.0.0:4318` |
| B | Tres destinos | **trazas → Tempo** (`otlp_grpc`, `tempo:4317`, sin TLS) · **logs → Loki** (`otlp_http`, `http://loki:3100/otlp`) · **métricas → Prometheus** (`otlp_http`, `http://prometheus:9090/api/v1/otlp`) |
| C | Protección | `memory_limiter` primero, con límite duro > 0 y margen de picos ≤ 25 % del límite |
| D | Ruido | descartar los spans llamados **`GET /healthz`** |
| E | PII | `customer.email` **seudonimizado (hash)** y `debug.prompt` **eliminado**, en **trazas y logs** |
| F | Entorno | `deployment.environment.name = lab-student` en **las tres señales** |
| G | Migración (módulo 04) | en spans: `gen_ai.system` → `gen_ai.provider.name` y `gen_ai.usage.prompt_tokens` → `gen_ai.usage.input_tokens` (copiar y borrar el antiguo) |
| H | Resiliencia | cada exportador de red con `sending_queue` (`queue_size ≥ 1000`) y `retry_on_failure` (`max_elapsed_time ≥ 60s`) |
| I | Operación | `health_check` activo y métricas internas del Collector en `:8888`; `batch` en todos los pipelines |

## Datos

La **telemetría de prueba** la genera `send_probe(endpoint)`: 5 spans `chat mock-small` (con `customer.email`, `debug.prompt`, `gen_ai.system`, `gen_ai.usage.prompt_tokens`), 3 spans `GET /healthz`, 2 logs con `customer.email` y un contador `supportops.probe.requests`. Es **determinista** (cantidades fijas): no depende de tu máquina.
Cadena de resiliencia: stack local → contenedor de laboratorio con exportador `file` (**sin** backends) → análisis estático del YAML (linter) si no hay Docker.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución, Collector contrib 0.162.0) |
|---|---|---|---|
| R1 | YAML válido | `cfglint` sin **errores** | 0 errores |
| R2 | Buenas prácticas | `cfglint` sin **avisos** (nada definido y sin usar, orden correcto, `memory_limiter`…) | 0 avisos (la solución tampoco tiene informativos) |
| R3 | Receptor | `otlp` con gRPC `0.0.0.0:4317` y HTTP `0.0.0.0:4318` | cumple |
| R4 | Tres pipelines y destinos | traces→`otlp_grpc` a `tempo:4317`; logs→`otlp_http` a `loki:3100/otlp`; metrics→`otlp_http` a `prometheus:9090` | 3/3 |
| R5 | Orden | en los 3 pipelines `memory_limiter` primero y `batch` último; en trazas el filtro va antes de los procesadores de atributos | cumple |
| R6 | `memory_limiter` | `limit_mib > 0` y `spike_limit_mib ≤ 25 %` de `limit_mib` | cumple (solución: 400 / 80 = 20 %) |
| R7 | Resiliencia | cada exportador usado: `queue_size ≥ 1000` y `max_elapsed_time ≥ 60 s` | 3/3 exportadores |
| R8 | Operación | `health_check` en `service.extensions` y lector `pull` de métricas internas en el puerto 8888 | cumple |
| R9 | `otelcol validate` | La configuración pasa `validate` con la imagen 0.162.0 (**requiere Docker**) | exit 0 |
| R10 | Comportamiento | Con `send_probe` y exportadores `file`: 5 spans a la salida (0 de salud), 0 correos en claro y 0 `debug.prompt` en spans y logs, renombres aplicados, entorno en trazas, logs y métricas (**requiere Docker**) | todos los controles |
| R11 | Extremo a extremo | Tu config, conectada al stack, hace llegar los datos de prueba a **Tempo, Loki y Prometheus** (**requiere el stack**) | 3/3 |

> 📏 R1–R8 son análisis del YAML; R9–R11 ejecutan el Collector real. Los umbrales son lo que mide la solución de referencia. R10 es **funcional**: da igual *cómo* lo escribas, importa que la telemetría que sale sea la correcta.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import json, re, shutil, subprocess, tempfile, time, uuid, logging, collections, pathlib, os, urllib.request, urllib.error
import yaml
from opentelemetry.trace import SpanKind
from opentelemetry._logs import SeverityNumber
from supportops.telemetry import setup_telemetry

STACK = check_stack(["tempo", "loki", "prometheus"])
logging.getLogger("opentelemetry").setLevel(logging.CRITICAL)

IMAGE = "otel/opentelemetry-collector-contrib:0.162.0"        # la misma imagen (y versión) que usa el stack del curso

def docker_ok() -> bool:
    try:
        return subprocess.run(["docker", "info"], capture_output=True, timeout=15).returncode == 0
    except Exception:
        return False

INTERNAL_TELEMETRY = """  telemetry:
    metrics:
      level: normal
      readers:
        - pull:
            exporter:
              prometheus:
                host: 0.0.0.0
                port: 8888
"""

class LabCollector:
    """Collector de laboratorio en un contenedor efímero con puertos de host ALEATORIOS (no choca con el stack del curso).
    Monta `config` y un directorio /out (para el exportador `file`). `with LabCollector(cfg) as lab:` -> lab.otlp_http, lab.read(...), lab.internal_metrics()."""
    def __init__(self, config: str, network: str | None = None):
        self.config, self.network = config, network
        self.dir = pathlib.Path(tempfile.mkdtemp(prefix="otelcol-lab-")); (self.dir / "out").mkdir(); os.chmod(self.dir / "out", 0o777)
        (self.dir / "config.yaml").write_text(config); os.chmod(self.dir / "config.yaml", 0o644)
        self.name = f"otelcol-lab-{uuid.uuid4().hex[:8]}"
    def __enter__(self):
        cmd = ["docker", "run", "-d", "--name", self.name, "-p", "127.0.0.1::4318", "-p", "127.0.0.1::4317", "-p", "127.0.0.1::8888",
               "-v", f"{self.dir / 'config.yaml'}:/etc/otelcol-contrib/config.yaml:ro", "-v", f"{self.dir / 'out'}:/out"]
        if self.network: cmd += ["--network", self.network]
        r = subprocess.run(cmd + [IMAGE, "--config=/etc/otelcol-contrib/config.yaml"], capture_output=True, text=True)
        if r.returncode: raise RuntimeError(r.stderr[:500])
        self.port = {p: int(subprocess.run(["docker", "port", self.name, f"{p}/tcp"], capture_output=True, text=True).stdout.split(":")[-1]) for p in (4318, 4317, 8888)}
        for _ in range(80):
            if "Everything is ready" in self.logs() or self.exited(): break
            time.sleep(0.25)
        if self.exited():
            msg = self.logs()[-800:]; self.__exit__()                       # limpia el contenedor fallido antes de informar
            raise RuntimeError("el Collector no arrancó:\n" + msg)
        return self
    def exited(self) -> bool:
        return subprocess.run(["docker", "inspect", "-f", "{{.State.Running}}", self.name], capture_output=True, text=True).stdout.strip() != "true"
    def logs(self) -> str:
        r = subprocess.run(["docker", "logs", self.name], capture_output=True, text=True); return r.stdout + r.stderr
    @property
    def otlp_http(self) -> str: return f"http://127.0.0.1:{self.port[4318]}"
    def internal_metrics(self) -> dict:
        """Métricas internas del Collector (otelcol_*) sumadas por nombre. Requiere el bloque `service.telemetry` (INTERNAL_TELEMETRY)."""
        txt = urllib.request.build_opener(urllib.request.ProxyHandler({})).open(f"http://127.0.0.1:{self.port[8888]}/metrics", timeout=5).read().decode()
        out: dict = {}
        for ln in txt.splitlines():
            if ln.startswith("otelcol_"):
                name, val = ln.rsplit(" ", 1); name = re.sub(r"\{.*\}", "", name); out[name] = out.get(name, 0.0) + float(val)
        return out
    def read(self, fname: str) -> list:
        p = self.dir / "out" / fname
        return [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []
    def __exit__(self, *a):
        subprocess.run(["docker", "rm", "-f", self.name], capture_output=True); shutil.rmtree(self.dir, ignore_errors=True)

def _kv(attrs): return {a["key"]: next(iter(a["value"].values())) for a in attrs}
def flat_spans(lines):
    """Líneas OTLP/JSON del exportador `file` -> lista de spans con su resource aplanado."""
    return [{"name": s["name"], "attrs": _kv(s.get("attributes", [])), "resource": _kv(rs["resource"].get("attributes", []))}
            for l in lines for rs in l.get("resourceSpans", []) for ss in rs["scopeSpans"] for s in ss["spans"]]
def flat_logs(lines):
    return [{"body": next(iter(r["body"].values())) if r.get("body") else "", "attrs": _kv(r.get("attributes", [])), "resource": _kv(rs["resource"].get("attributes", []))}
            for l in lines for rs in l.get("resourceLogs", []) for sl in rs["scopeLogs"] for r in sl["logRecords"]]
def flat_metrics(lines):
    return [{"name": m["name"], "resource": _kv(rs["resource"].get("attributes", []))} for l in lines for rs in l.get("resourceMetrics", []) for sm in rs["scopeMetrics"] for m in sm["metrics"]]

def send_probe(endpoint: str, n_chat: int = 5, n_health: int = 3, n_logs: int = 2) -> dict:
    """Envía telemetría de PRUEBA con los defectos que un Collector bien configurado debe corregir: PII (customer.email), un atributo de depuración
    (debug.prompt), nombres antiguos (gen_ai.system, gen_ai.usage.prompt_tokens) y ruido (spans «GET /healthz»). Devuelve lo enviado."""
    tel = setup_telemetry("supportops-probe", endpoint=endpoint, mode="otlp", set_global=False, metric_interval_s=1, keep_local=False)
    tr, lg = tel.tracer(), tel.logger()
    for i in range(n_chat):
        with tr.start_as_current_span("chat mock-small", kind=SpanKind.CLIENT, attributes={"gen_ai.operation.name": "chat", "gen_ai.system": "mock",
                "gen_ai.usage.prompt_tokens": 100 + i, "customer.email": f"cliente{i}@example.com", "debug.prompt": "texto largo de depuración"}): pass
    for _ in range(n_health):
        with tr.start_as_current_span("GET /healthz"): pass
    for i in range(n_logs):
        lg.emit(body="ticket recibido", severity_number=SeverityNumber.INFO, severity_text="INFO", attributes={"customer.email": f"cliente{i}@example.com"})
    tel.meter().create_counter("supportops.probe.requests").add(7)
    tel.force_flush(10); time.sleep(1.5); tel.force_flush(10); time.sleep(1.0)
    tel.shutdown()
    return {"chat": n_chat, "health": n_health, "logs": n_logs}

def to_file_config(cfg_text: str) -> str:
    """Deriva una config de PRUEBA de otra: mismos receptores/procesadores/pipelines, pero los exportadores pasan a ser `file` (sin backends)."""
    cfg = yaml.safe_load(cfg_text)
    cfg["exporters"] = {f"file/{s}": {"path": f"/out/{s}.jsonl"} for s in ("traces", "metrics", "logs")}
    for name, p in cfg["service"]["pipelines"].items(): p["exporters"] = [f"file/{name.split('/')[0]}"]
    tele = cfg["service"].setdefault("telemetry", {}); tele.setdefault("metrics", {"level": "normal", "readers": [{"pull": {"exporter": {"prometheus": {"host": "0.0.0.0", "port": 8888}}}}]})
    return yaml.safe_dump(cfg, sort_keys=False)

DOCKER = docker_ok()
print("Docker disponible:", DOCKER)

## Herramientas que ya tienes (de la lección)

- `cfglint(texto_yaml)` — linter offline (reglas `UNKNOWN`, `UNDEF`, `SIGNAL`, `ORDER`, `UNUSED`…).
- `otelcol_validate(texto_yaml)` — `otelcol validate` en un contenedor (definido abajo).
- `LabCollector(config)` — Collector de laboratorio; `send_probe(endpoint)` — telemetría de prueba; `to_file_config(cfg)` — la misma config con exportadores `file`.

In [ ]:
COMP = json.loads((ROOT / "05_collector" / "collector_components_0.162.0.json").read_text())      # componentes de la distribución contrib 0.162.0 y su estabilidad por señal
SECTIONS = ("receivers", "processors", "exporters", "connectors", "extensions")
ALIASES = {"otlphttp": "otlp_http", "prometheusremotewrite": "prometheus_remote_write", "spanmetrics": "span_metrics", "servicegraph": "service_graph", "k8sattributes": "k8s_attributes",
           "resourcedetection": "resource_detection", "filelog": "file_log", "hostmetrics": "host_metrics", "loadbalancing": "load_balancing", "metricstransform": "metrics_transform"}
EXPORTER_ALIAS = {"otlp": "otlp_grpc"}                                   # como EXPORTADOR, `otlp` es el nombre antiguo de otlp_grpc (como receptor sigue siendo `otlp`)
CFG_SEVERITY = {"PARSE": "error", "NOPIPE": "error", "UNKNOWN": "error", "UNDEF": "error", "SIGNAL": "error", "MLCFG": "error",
                "ALIAS": "warn", "UNUSED": "warn", "ORDER": "warn", "NOML": "warn", "EXPQ": "warn", "BIND": "warn", "SECRET": "warn", "NULL": "warn",
                "NOBATCH": "info", "HEALTH": "info"}
PHASE = {"memory_limiter": 1, "filter": 2, "probabilistic_sampler": 2, "tail_sampling": 2, "k8s_attributes": 3, "resource_detection": 3,
         "attributes": 4, "resource": 4, "transform": 4, "redaction": 4, "span": 4, "group_by_attrs": 4, "metrics_transform": 4, "batch": 5}
NETWORK_EXPORTERS = {"otlp_grpc", "otlp_http", "otlp", "otlphttp", "prometheus_remote_write", "prometheusremotewrite", "kafka", "load_balancing"}

def ctype(key: str) -> str: return key.split("/", 1)[0]

def cfglint(cfg_text: str) -> list:
    """Hallazgos [{rule, severity, where, msg}] de una configuración del Collector SIN ejecutar el Collector."""
    F = []
    def add(rule, where, msg): F.append({"rule": rule, "severity": CFG_SEVERITY[rule], "where": where, "msg": msg})
    try:
        cfg = yaml.safe_load(cfg_text); assert isinstance(cfg, dict)
    except Exception as e:
        add("PARSE", "yaml", f"no es un YAML de mapa válido: {str(e)[:80]}"); return F
    for sec in SECTIONS:                                                                   # 1) secciones nulas («processors:» vacío)
        if sec in cfg and cfg[sec] is None: add("NULL", sec, f"«{sec}:» vacío (null); usa {{}} o elimínalo")
    defined = {s: dict(cfg.get(s) or {}) for s in SECTIONS}
    for sec, items in defined.items():                                                     # 2) tipos de componente que existen en la distribución
        for key in items:
            t = ctype(key)
            if t in COMP[sec]: continue
            alias = ALIASES.get(t) or (EXPORTER_ALIAS.get(t) if sec == "exporters" else None)
            if alias and alias in COMP[sec]: add("ALIAS", f"{sec}.{key}", f"nombre antiguo: el canónico hoy es «{alias}» (puede aceptarse como alias; no cuentes con ello)")
            else: add("UNKNOWN", f"{sec}.{key}", f"«{t}» no existe en la distribución contrib {COMP['version']}")
    svc = cfg.get("service") or {}; pipes = svc.get("pipelines") or {}
    if not pipes: add("NOPIPE", "service.pipelines", "el servicio necesita al menos un pipeline")
    used = {s: set() for s in SECTIONS}
    for pname, p in pipes.items():                                                         # 3) referencias, señales y orden
        sig = ctype(pname); p = p or {}
        for sec in ("receivers", "processors", "exporters"):
            for key in p.get(sec) or []:
                used[sec].add(key)
                if key not in defined[sec]:
                    add("UNDEF", f"pipelines.{pname}.{sec}", f"referencia a «{key}», que no está configurado en «{sec}»"); continue
                t = ALIASES.get(ctype(key), EXPORTER_ALIAS.get(ctype(key), ctype(key)) if sec == "exporters" else ctype(key))
                st = COMP[sec].get(t)
                if st and sig in ("traces", "metrics", "logs") and sig not in st: add("SIGNAL", f"pipelines.{pname}.{sec}", f"«{key}» no soporta la señal {sig}")
        procs = [ctype(k) for k in p.get("processors") or []]
        if "memory_limiter" not in procs: add("NOML", f"pipelines.{pname}", "sin memory_limiter: el Collector no se protege de quedarse sin memoria")
        ph = [(PHASE[t], t) for t in procs if t in PHASE]
        for (a, ta), (b, tb) in zip(ph, ph[1:]):
            if a > b: add("ORDER", f"pipelines.{pname}.processors", f"«{tb}» debería ir antes que «{ta}» (memory_limiter → filtros/muestreo → contexto → enriquecimiento → batch)")
        has_q_batch = any(((defined["exporters"].get(k) or {}).get("sending_queue") or {}).get("batch") is not None for k in p.get("exporters") or [])
        if "batch" not in procs and not has_q_batch: add("NOBATCH", f"pipelines.{pname}", "sin batch (ni procesador ni sending_queue.batch): más peticiones y peor compresión")
    for sec in ("receivers", "processors", "exporters"):
        for key in defined[sec]:
            if key not in used[sec]: add("UNUSED", f"{sec}.{key}", "definido pero no usado en ningún pipeline: el Collector lo ignora en silencio")
    for key, c in defined["processors"].items():                                           # 4) configuración de componentes concretos
        if ctype(key) == "memory_limiter" and not ((c or {}).get("limit_mib") or (c or {}).get("limit_percentage")):
            add("MLCFG", f"processors.{key}", "memory_limiter sin limit_mib ni limit_percentage (el Collector lo rechaza)")
    for key, c in defined["exporters"].items():
        if ctype(key) in NETWORK_EXPORTERS and (((c or {}).get("sending_queue") or {}).get("enabled") is False):
            add("EXPQ", f"exporters.{key}", "sending_queue desactivada en un exportador de red: sin búfer ante caídas del backend")
    exts = svc.get("extensions") or []
    for e in exts:
        if e not in defined["extensions"]: add("UNDEF", "service.extensions", f"referencia a «{e}», que no está configurada")
    if not any(ctype(e) == "health_check" for e in exts): add("HEALTH", "service.extensions", "sin health_check: no hay endpoint de salud para sondas")
    for key, c in defined["extensions"].items():
        if ctype(key) in ("pprof", "zpages") and "0.0.0.0" in json.dumps(c or {}): add("BIND", f"extensions.{key}", "pprof/zpages escuchando en 0.0.0.0: expónlo solo en localhost")
    for m in re.finditer(r"(?im)^\s*(authorization|api[_-]?key|password|token|secret)\s*:\s*(?!\$\{)(\S{6,})", cfg_text):
        add("SECRET", m.group(1), "valor literal con aspecto de credencial: usa ${env:NOMBRE}")
    return F

def otelcol_validate(cfg_text: str) -> tuple[bool, str]:
    d = pathlib.Path(tempfile.mkdtemp()); (d / "c.yaml").write_text(cfg_text); os.chmod(d / "c.yaml", 0o644)
    r = subprocess.run(["docker", "run", "--rm", "-v", f"{d / 'c.yaml'}:/c.yaml:ro", IMAGE, "validate", "--config=/c.yaml"], capture_output=True, text=True)
    shutil.rmtree(d, ignore_errors=True)
    return r.returncode == 0, " ".join((r.stdout + r.stderr).split())[:300]

## Paso 1 · Escribe la configuración

**TODO 1.** Completa `CONFIG` (un YAML en una cadena) para cumplir los requisitos A-I. El esqueleto trae el receptor y la extensión; **faltan** los procesadores, los exportadores y los pipelines (y el bloque `service`).
Aplica lo aprendido: el **orden** de la lección (sección 2), los nombres canónicos 0.162 (`otlp_grpc`, `otlp_http`), y las sintaxis de `filter` (condición OTTL sobre el nombre), `attributes` (acciones `hash`/`delete`), `resource` (`upsert`) y `transform` (OTTL con `set(...)` y `delete_key(...)`).

<details>
<summary>🪜 Pista 1</summary>

Procesadores que necesitas (en este orden en los tres pipelines cuando apliquen): `memory_limiter` → `filter/health` (solo trazas) → `attributes/pii` (trazas y logs) → `resource/lab` (las tres) → `transform/semconv` (solo trazas) → `batch`. Cada uno se **define** en `processors:` y se **activa** en el pipeline.

</details>

<details>
<summary>🪜 Pista 2</summary>

Sintaxis (compruébala con `cfglint` y `otelcol_validate`):
```yaml
filter/health:
  error_mode: ignore
  traces:
    span:
      - 'name == "GET /healthz"'
attributes/pii:
  actions:
    - {key: customer.email, action: hash}
    - {key: debug.prompt, action: delete}
resource/lab:
  attributes:
    - {key: deployment.environment.name, value: lab-student, action: upsert}
```

</details>

<details>
<summary>🪜 Pista 3</summary>

Renombrar en spans con OTTL (copiar y borrar):
```yaml
transform/semconv:
  error_mode: ignore
  trace_statements:
    - set(span.attributes["gen_ai.provider.name"], span.attributes["gen_ai.system"]) where span.attributes["gen_ai.system"] != nil
    - delete_key(span.attributes, "gen_ai.system")
```
Y lo mismo con `gen_ai.usage.prompt_tokens` → `gen_ai.usage.input_tokens`.

</details>

<details>
<summary>🪜 Pista 4</summary>

Exportadores (uno por backend, con cola y reintentos explícitos):
```yaml
otlp_grpc/tempo:
  endpoint: tempo:4317
  tls: {insecure: true}
  sending_queue: {enabled: true, queue_size: 2000}
  retry_on_failure: {enabled: true, max_elapsed_time: 120s}
otlp_http/loki:        {endpoint: "http://loki:3100/otlp", ...}
otlp_http/prometheus:  {endpoint: "http://prometheus:9090/api/v1/otlp", ...}
```
Métricas internas: `service.telemetry.metrics.readers: [{pull: {exporter: {prometheus: {host: 0.0.0.0, port: 8888}}}}]`.

</details>

In [ ]:
CONFIG = """# Collector del alumno · gateway del laboratorio SupportOps
receivers:
  otlp:
    protocols:
      grpc:
        endpoint: 0.0.0.0:4317
      http:
        endpoint: 0.0.0.0:4318

processors: {}        # TODO 1a: memory_limiter, filter/health, attributes/pii, resource/lab, transform/semconv, batch

exporters: {}         # TODO 1b: otlp_grpc/tempo, otlp_http/loki, otlp_http/prometheus (con sending_queue y retry_on_failure)

extensions:
  health_check:
    endpoint: 0.0.0.0:13133

service:
  extensions: [health_check]
  pipelines: {}       # TODO 1c: traces, logs y metrics (receivers, processors EN ORDEN, exporters) y service.telemetry para las métricas internas en :8888
"""
print(len(CONFIG.splitlines()), "líneas")

## Paso 2 · Comprueba sin ejecutar nada: el linter

Itera hasta que no queden **errores** ni **avisos** (los de nivel *info* son opcionales).

In [ ]:
def show_lint(cfg):
    F = cfglint(cfg)
    df = pd.DataFrame(F, columns=["rule", "severity", "where", "msg"]) if F else pd.DataFrame(columns=["rule", "severity", "where", "msg"])
    print(f"{(df.severity == 'error').sum()} errores · {(df.severity == 'warn').sum()} avisos · {(df.severity == 'info').sum()} informativos")
    return df
show_lint(CONFIG)

## Paso 3 · `otelcol validate` (requiere Docker)

In [ ]:
if DOCKER:
    ok, msg = otelcol_validate(CONFIG); print("otelcol validate:", "✅ OK" if ok else "❌ " + msg)
else:
    print("Sin Docker: salta este paso (el linter del paso 2 es tu red de seguridad).")

## Paso 4 · Comprueba el comportamiento sobre telemetría real (requiere Docker)

Ejecutamos **tu configuración** con los exportadores sustituidos por `file` (`to_file_config`: mismos receptores, procesadores y pipelines), enviamos la telemetría de prueba y leemos lo que **sale**:

In [ ]:
def functional_test(cfg: str) -> dict:
    with LabCollector(to_file_config(cfg)) as lab:
        send_probe(lab.otlp_http, n_chat=5, n_health=3, n_logs=2)
        return {"spans": flat_spans(lab.read("traces.jsonl")), "logs": flat_logs(lab.read("logs.jsonl")), "metrics": flat_metrics(lab.read("metrics.jsonl"))}

if DOCKER:
    try:
        FT = functional_test(CONFIG)
        print(f"salen {len(FT['spans'])} spans, {len(FT['logs'])} logs, {len(FT['metrics'])} métricas")
        if FT["spans"]: print("ejemplo de span:", FT["spans"][0]["name"], {k: (v[:10] + '…' if isinstance(v, str) and len(v) > 10 else v) for k, v in FT["spans"][0]["attrs"].items()})
    except Exception as e:
        FT = None; print("No se pudo ejecutar (¿falta completar la configuración?):", str(e)[:300])
else:
    FT = None; print("Sin Docker: salta este paso.")

## Paso 5 · Extremo a extremo con el stack (opcional)

Con el stack arriba (`docker compose --profile core --profile metrics up -d`), **tu** Collector se conecta a la red de Compose (`agentobs_default`: ahí se resuelven `tempo`, `loki` y `prometheus`), recibe la telemetría de prueba y la entrega. Comprobamos que **llega a los tres backends**.
Es **lectura**: no tocamos el stack, y tu Collector usa puertos aleatorios de tu máquina (el del stack sigue intacto).

In [ ]:
def end_to_end(cfg: str, wait_s: int = 60) -> dict:
    res = {"tempo": 0, "loki": 0, "prometheus": 0.0}
    with LabCollector(cfg, network="agentobs_default") as lab:
        send_probe(lab.otlp_http, n_chat=5, n_health=3, n_logs=2)
        t0 = time.time()
        while time.time() - t0 < wait_s and not (res["tempo"] >= 5 and res["loki"] >= 2 and res["prometheus"] >= 7):
            try:
                res["tempo"] = len(backends.tempo_search('{ resource.service.name = "supportops-probe" && span.gen_ai.provider.name = "mock" }', minutes=15, limit=50).get("traces", []))
                res["loki"] = len(backends.loki_query('{service_name="supportops-probe"} |= "ticket recibido"', minutes=15, limit=50))
                pr = backends.prom_query('sum(supportops_probe_requests_total{job="supportops-probe"})'); res["prometheus"] = float(pr[0]["value"][1]) if pr else 0.0
            except backends.BackendError:
                pass
            time.sleep(3)
    return res

if STACK.degraded or not DOCKER:
    E2E = None; print("Modo degradado: levanta el stack (", STACK.command, ") y vuelve a ejecutar.")
else:
    try:
        E2E = end_to_end(CONFIG); print("llegado a los backends:", E2E)
    except Exception as e:
        E2E = None; print("No se pudo ejecutar:", str(e)[:300])

## Paso 6 · Rúbrica

In [ ]:
def _dur_s(v) -> float:
    m = re.fullmatch(r"(\d+(?:\.\d+)?)(ms|s|m|h)", str(v).strip()); return float(m[1]) * {"ms": 1e-3, "s": 1, "m": 60, "h": 3600}[m[2]] if m else 0.0

def run_rubric():
    rows = []
    def add(r, desc, ok, detail=""): rows.append({"criterio": r, "descripción": desc, "resultado": ("✅ OK" if ok else "❌ FALLA") if ok is not None else "⏭ no evaluable", "detalle": detail})
    F = cfglint(CONFIG); errs = [f for f in F if f["severity"] == "error"]; warns = [f for f in F if f["severity"] == "warn"]
    add("R1", "cfglint sin errores", not errs, "; ".join(f"{f['rule']}@{f['where']}" for f in errs[:3]))
    add("R2", "cfglint sin avisos", not warns and not errs, "; ".join(f"{f['rule']}@{f['where']}" for f in warns[:3]))
    try: cfg = yaml.safe_load(CONFIG) or {}
    except Exception: cfg = {}
    ex, pr, pipes = cfg.get("exporters") or {}, cfg.get("processors") or {}, (cfg.get("service") or {}).get("pipelines") or {}
    proto = (((cfg.get("receivers") or {}).get("otlp") or {}).get("protocols") or {})
    add("R3", "receptor otlp gRPC 4317 + HTTP 4318 en 0.0.0.0", (proto.get("grpc") or {}).get("endpoint") == "0.0.0.0:4317" and (proto.get("http") or {}).get("endpoint") == "0.0.0.0:4318")
    def pipe(sig): return next((p for n, p in pipes.items() if ctype(n) == sig), None) or {}
    def exps(sig): return [(ctype(k), str((ex.get(k) or {}).get("endpoint", ""))) for k in pipe(sig).get("exporters") or [] if k in ex]
    ok4 = {"traces": any(t == "otlp_grpc" and "tempo:4317" in e for t, e in exps("traces")),
           "logs": any(t == "otlp_http" and "loki:3100/otlp" in e for t, e in exps("logs")),
           "metrics": any(t in ("otlp_http", "prometheus_remote_write") and "prometheus:9090" in e for t, e in exps("metrics"))}
    add("R4", "traces→Tempo, logs→Loki, metrics→Prometheus", all(ok4.values()), str(ok4))
    def chain(sig): return [ctype(k) for k in pipe(sig).get("processors") or []]
    order_ok = all(chain(s)[:1] == ["memory_limiter"] and chain(s)[-1:] == ["batch"] for s in ("traces", "logs", "metrics"))
    ct = chain("traces"); order_ok = order_ok and "filter" in ct and all(ct.index("filter") < ct.index(x) for x in ("attributes", "resource", "transform") if x in ct)
    add("R5", "memory_limiter primero, batch último, filtro antes de enriquecer", order_ok, " | ".join(f"{s}: {chain(s)}" for s in ("traces", "logs", "metrics")))
    ml = next((c for k, c in pr.items() if ctype(k) == "memory_limiter"), None) or {}
    add("R6", "memory_limiter: limit_mib > 0 y spike ≤ 25 %", bool(ml.get("limit_mib")) and ml.get("spike_limit_mib", 0) <= 0.25 * ml.get("limit_mib", 0), f"limit={ml.get('limit_mib')} spike={ml.get('spike_limit_mib')}")
    used_ex = {k for s in ("traces", "logs", "metrics") for k in pipe(s).get("exporters") or [] if k in ex}
    good = [k for k in used_ex if ((ex[k] or {}).get("sending_queue") or {}).get("queue_size", 0) >= 1000 and _dur_s(((ex[k] or {}).get("retry_on_failure") or {}).get("max_elapsed_time")) >= 60]
    add("R7", "cada exportador: queue_size ≥ 1000 y max_elapsed_time ≥ 60 s", bool(used_ex) and len(good) == len(used_ex) and len(used_ex) >= 3, f"{len(good)}/{len(used_ex)} exportadores")
    readers = ((((cfg.get("service") or {}).get("telemetry") or {}).get("metrics") or {}).get("readers")) or []
    ok_int = any(((r.get("pull") or {}).get("exporter") or {}).get("prometheus", {}).get("port") == 8888 for r in readers if isinstance(r, dict))
    add("R8", "health_check + métricas internas en :8888", any(ctype(e) == "health_check" for e in (cfg.get("service") or {}).get("extensions") or []) and ok_int)
    if DOCKER:
        ok, msg = otelcol_validate(CONFIG); add("R9", "otelcol validate", ok, "" if ok else msg[:120])
        try:
            T = functional_test(CONFIG); sp, lg, mt = T["spans"], T["logs"], T["metrics"]
            checks = {"5 spans (0 healthz)": len(sp) == 5 and not any(s["name"] == "GET /healthz" for s in sp),
                      "sin correo en claro (spans y logs)": all("@" not in str(x["attrs"].get("customer.email", "")) for x in sp + lg) and bool(sp) and bool(lg),
                      "sin debug.prompt": not any("debug.prompt" in x["attrs"] for x in sp + lg),
                      "renombres": bool(sp) and all("gen_ai.system" not in s["attrs"] and s["attrs"].get("gen_ai.provider.name") == "mock" and "gen_ai.usage.prompt_tokens" not in s["attrs"] and "gen_ai.usage.input_tokens" in s["attrs"] for s in sp),
                      "entorno en trazas, logs y métricas": bool(sp) and bool(lg) and bool(mt) and all(x["resource"].get("deployment.environment.name") == "lab-student" for x in sp + lg + mt)}
            add("R10", "comportamiento sobre telemetría real", all(checks.values()), ", ".join(k for k, v in checks.items() if not v) or "todos los controles")
        except Exception as e:
            add("R10", "comportamiento sobre telemetría real", False, str(e)[:120])
    else:
        add("R9", "otelcol validate", None, "sin Docker"); add("R10", "comportamiento", None, "sin Docker")
    if DOCKER and not STACK.degraded:
        try:
            e = end_to_end(CONFIG, wait_s=45); add("R11", "llega a Tempo, Loki y Prometheus", e["tempo"] >= 5 and e["loki"] >= 2 and e["prometheus"] >= 7, str(e))
        except Exception as e:
            add("R11", "llega a los tres backends", False, str(e)[:120])
    else:
        add("R11", "llega a los tres backends", None, "sin stack/Docker")
    return pd.DataFrame(rows).set_index("criterio")

display(run_rubric())

---
## Retos extra (nivel experto)

1. **Métricas RED desde las trazas:** añade el **conector** `span_metrics` (sale de un pipeline de trazas y entra en uno de métricas) y comprueba en Prometheus las métricas de llamadas y duración por servicio. ¿Qué pasa con su cardinalidad si pones `customer.email` como dimensión?
2. **Cola persistente:** añade la extensión `file_storage` y úsala en `sending_queue.storage` del exportador de Tempo; reinicia el Collector con datos en cola y demuestra que no se pierden.
3. **Agente → gateway:** escribe una segunda configuración de **agente** (receptor OTLP, `resource_detection`, exportador `otlp_grpc` hacia tu gateway) y encadénalos con `LabCollector`. ¿Qué procesadores pondrías en cada capa y por qué?
4. **Variables de entorno:** parametriza endpoints y límites con `${env:TEMPO_ENDPOINT:-tempo:4317}` y `${env:LIMIT_MIB:-400}`; ¿cómo las inyectarías en Compose y en Kubernetes?
5. **Alerta:** escribe la expresión PromQL de una alerta sobre las métricas internas que avise cuando la cola de un exportador supere el 70 % durante 5 minutos (`otelcol_exporter_queue_size / otelcol_exporter_queue_capacity`).

## Reflexión (producción e hilo conductor)

- Tu `attributes/pii` aplica `hash` al correo. ¿Qué garantía da realmente un hash SHA-256 **sin sal** sobre un espacio de correos acotado? ¿Qué alternativa (sal secreta, tokenización, eliminar) elegirías y qué perdería la correlación?
- Has puesto la **política de PII en el gateway**. ¿Qué ocurre con los datos *antes* de llegar a él (agente, red, logs del propio Collector con el exportador `debug`)?
- El `filter/health` descarta spans por nombre exacto. ¿Qué pasaría si un equipo cambia el nombre a `GET /health`? ¿Cómo detectarías **datos que desaparecen** por un filtro (pista: `otelcol_processor_incoming_items` frente a `outgoing_items`)?
- En el módulo 18 sustituirás la «cola y reintentos» por **muestreo**; en el 19 desplegarás este gateway con Helm. ¿Qué parámetros de tu configuración deberían venir de variables de entorno o de *values*?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa (la verificada con el linter, `otelcol validate` y los controles funcionales) y la rúbrica con ella.

In [ ]:
CONFIG = """# Collector del alumno · gateway del laboratorio SupportOps
receivers:
  otlp:
    protocols:
      grpc:
        endpoint: 0.0.0.0:4317
      http:
        endpoint: 0.0.0.0:4318

processors:
  memory_limiter:                 # 1.º: protege al Collector (límite duro 400 MiB, margen para picos 80 MiB = 20 %)
    check_interval: 1s
    limit_mib: 400
    spike_limit_mib: 80
  filter/health:                  # 2.º: descarta ruido ANTES de gastar CPU en el resto
    error_mode: ignore
    traces:
      span:
        - 'name == "GET /healthz"'
  attributes/pii:                 # PII: el correo se seudonimiza (hash) y el atributo de depuración se elimina
    actions:
      - key: customer.email
        action: hash
      - key: debug.prompt
        action: delete
  resource/lab:                   # etiqueta de entorno del laboratorio del alumno
    attributes:
      - key: deployment.environment.name
        value: lab-student
        action: upsert
  transform/semconv:              # migración a la spec GenAI actual (módulo 04): renombres en spans
    error_mode: ignore
    trace_statements:
      - set(span.attributes["gen_ai.provider.name"], span.attributes["gen_ai.system"]) where span.attributes["gen_ai.system"] != nil
      - delete_key(span.attributes, "gen_ai.system")
      - set(span.attributes["gen_ai.usage.input_tokens"], span.attributes["gen_ai.usage.prompt_tokens"]) where span.attributes["gen_ai.usage.prompt_tokens"] != nil
      - delete_key(span.attributes, "gen_ai.usage.prompt_tokens")
  batch:                          # último: agrupa lo que ya está filtrado y transformado
    timeout: 1s
    send_batch_size: 512

exporters:
  otlp_grpc/tempo:                # trazas -> Tempo (OTLP/gRPC dentro de la red de Compose)
    endpoint: tempo:4317
    tls:
      insecure: true
    sending_queue:
      enabled: true
      queue_size: 2000
    retry_on_failure:
      enabled: true
      max_elapsed_time: 120s
  otlp_http/loki:                 # logs -> Loki (endpoint OTLP nativo: el exportador añade /v1/logs)
    endpoint: http://loki:3100/otlp
    sending_queue:
      enabled: true
      queue_size: 2000
    retry_on_failure:
      enabled: true
      max_elapsed_time: 120s
  otlp_http/prometheus:           # métricas -> receptor OTLP de Prometheus (/api/v1/otlp/v1/metrics)
    endpoint: http://prometheus:9090/api/v1/otlp
    sending_queue:
      enabled: true
      queue_size: 2000
    retry_on_failure:
      enabled: true
      max_elapsed_time: 120s

extensions:
  health_check:
    endpoint: 0.0.0.0:13133

service:
  extensions: [health_check]
  telemetry:
    metrics:
      level: normal
      readers:
        - pull:
            exporter:
              prometheus:
                host: 0.0.0.0
                port: 8888
  pipelines:
    traces:
      receivers: [otlp]
      processors: [memory_limiter, filter/health, attributes/pii, resource/lab, transform/semconv, batch]
      exporters: [otlp_grpc/tempo]
    logs:
      receivers: [otlp]
      processors: [memory_limiter, attributes/pii, resource/lab, batch]
      exporters: [otlp_http/loki]
    metrics:
      receivers: [otlp]
      processors: [memory_limiter, resource/lab, batch]
      exporters: [otlp_http/prometheus]
"""

In [ ]:
# Re-ejecuta la rúbrica con la solución
display(run_rubric())